# IoT Network Attack Detection with a Neural Network (MLP)

<a href="https://colab.research.google.com/github/buildwithyang/net-attack-predict/blob/main/notebooks/iot_attack_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Problem.** An IoT network sees thousands of traffic records ("flows"). For each one we want to
say: is it **normal** traffic or an **attack**? We train a multilayer perceptron (MLP) to decide.

**How to use this notebook.** In Colab: *Runtime → Run all*. Everything is in this one file:
it downloads the dataset by itself, and every line of data processing and training is below.
Then go to **Step 7**, change a setting, and run that cell again to see what changes.

| Step | What happens | Course link |
| --- | --- | --- |
| 1 | Load and clean the data | |
| 2 | Split into train / validation / test | Week 2: generalisation, validation |
| 3 | Turn the table into numbers and scale them | Week 2: normalisation |
| 4 | Build the MLP | Week 1: perceptron → multilayer network |
| 5 | The training loop: forward → loss → backpropagation → update | Week 1–2: backpropagation, gradient descent |
| 6 | Measure the result on the test set | confusion matrix |
| 7 | **Experiment**: change settings, train, compare | Week 2: overfitting, early stopping, dropout |

## 0. Setup

Imports, and one fixed **random seed** (随机数种子). The seed makes the split, the starting weights and
the shuffling the same every time, so the same settings always give the same result.

In [ ]:
import copy
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn

SEED = 42
DATA_URL = "https://github.com/buildwithyang/net-attack-predict/raw/main/data/raw/rt-iot2022.zip"

## 1. Load and clean the data

Dataset: **RT-IoT2022** ([Kaggle](https://www.kaggle.com/datasets/supplejade/rt-iot2022real-time-internet-of-things)),
123,117 network flows, 83 features each, and a column `Attack_type` that says what the flow was.

Cleaning:
- drop `no` (just a row counter, not information about the traffic);
- remove exact duplicate rows (otherwise the same row could end up in both training and test);
- make the label: `Thing_Speak`, `MQTT_Publish`, `Wipro_bulb` are normal IoT devices → **0 = normal**;
  every other type is an attack → **1 = attack**.

In [ ]:
# Use the project copy of the data if it is there, otherwise download it (3.8 MB).
local = [Path("../data/raw/rt-iot2022.zip"), Path("data/raw/rt-iot2022.zip")]
zip_path = next((p for p in local if p.exists()), Path("rt-iot2022.zip"))
if not zip_path.exists():
    urllib.request.urlretrieve(DATA_URL, zip_path)

df = pd.read_csv(zip_path)                               # pandas can read the CSV inside the zip
df = df.drop(columns=["no"])
df["Attack_type"] = df["Attack_type"].str.strip()        # labels end with a hidden "\r" in the file
rows_before = len(df)
df = df.drop_duplicates()
print(f"{rows_before:,} rows, {rows_before - len(df):,} duplicates removed, {len(df):,} left")

NORMAL_TYPES = ["Thing_Speak", "MQTT_Publish", "Wipro_bulb"]
df["label"] = (~df["Attack_type"].isin(NORMAL_TYPES)).astype(int)   # 0 = normal, 1 = attack

counts = df.groupby("Attack_type")["label"].agg(rows="size", label="first").sort_values("rows", ascending=False)
counts["label"] = counts["label"].map({0: "normal", 1: "attack"})
display(counts)
print(f"attacks: {df['label'].mean():.1%} of all rows")

⚠️ About **90% of the rows are attacks**. A "model" that always answers *attack* would already be
90% accurate, so accuracy alone says little. We will look at two numbers instead:
- **detection rate**: of all attacks, how many did we catch? (higher is better)
- **false alarm rate**: of all normal traffic, how much did we wrongly flag as attack? (lower is better)

## 2. Split: train / validation / test

| Part | Share | Used for |
| --- | --- | --- |
| **train** | 70% | the network learns its weights from these rows |
| **validation** | 15% | checked after every epoch, to see if the network still improves on rows it did not learn from; decides when to stop |
| **test** | 15% | used **once at the very end**, for the final result |

The split is *stratified*: every attack type keeps the same share in all three parts.

In [ ]:
train_df, rest = train_test_split(df, train_size=0.70, stratify=df["Attack_type"], random_state=SEED)
val_df, test_df = train_test_split(rest, test_size=0.50, stratify=rest["Attack_type"], random_state=SEED)
parts = {"train": train_df, "val": val_df, "test": test_df}

for name, part in parts.items():
    print(f"{name:5s} {len(part):7,} rows   attacks {part['label'].mean():.1%}")

## 3. From table to numbers (features)

A neural network only works with numbers.
- `proto` and `service` are text (e.g. `tcp`, `mqtt`). **One-hot encoding** turns each value into its
  own 0/1 column.
- All other columns are numbers, but on very different scales: some are 0–1, some reach
  20,000,000,000. Big inputs make huge weighted sums, and training becomes unstable.
  **Scaling** (normalisation) fixes that. Three options, chosen in Step 7:
  - `standard`: subtract the mean, divide by the standard deviation → most values between about −3 and 3;
  - `log + standard`: first squash huge values with a logarithm, then standardise;
  - `none`: keep the raw numbers (to see why scaling matters).

The scaler learns its mean and standard deviation from the **training rows only**, then applies them to
validation and test. Otherwise information from the test set would leak into training.

In [ ]:
CATEGORICAL = ["proto", "service"]
NUMERIC = [c for c in df.columns if c not in CATEGORICAL + ["Attack_type", "label"]]
ONEHOT_COLUMNS = pd.get_dummies(train_df[CATEGORICAL]).columns   # categories seen in training


def prepare_data(scaling):
    # Returns X (inputs) and y (labels) for train / val / test as PyTorch tensors.
    numbers = {name: part[NUMERIC].to_numpy(dtype=float) for name, part in parts.items()}
    if scaling == "log + standard":
        numbers = {name: np.sign(v) * np.log1p(np.abs(v)) for name, v in numbers.items()}
    if scaling in ("standard", "log + standard"):
        scaler = StandardScaler().fit(numbers["train"])          # learn mean / std from TRAIN only
        numbers = {name: scaler.transform(v) for name, v in numbers.items()}

    X, y = {}, {}
    for name, part in parts.items():
        onehot = pd.get_dummies(part[CATEGORICAL]).reindex(columns=ONEHOT_COLUMNS, fill_value=0)
        X[name] = torch.tensor(np.hstack([numbers[name], onehot.to_numpy(dtype=float)]), dtype=torch.float32)
        y[name] = torch.tensor(part["label"].to_numpy(), dtype=torch.float32)
    return X, y


X, y = prepare_data("standard")
print("X train:", tuple(X["train"].shape), "→", X["train"].shape[1], "input features per row")
# Two example columns, before and after scaling (training rows)
for column in ["fwd_pkts_tot", "flow_iat.tot"]:
    before, after = train_df[column], X["train"][:, NUMERIC.index(column)]
    print(f"{column:14s} before: {before.min():>8,.0f} … {before.max():>16,.0f}    after: {after.min():6.2f} … {after.max():6.2f}")

## 4. The model: a multilayer perceptron (MLP)

Week 1 started with **one perceptron**: weighted sum of the inputs + bias → activation function.
An MLP stacks layers of them:

```
inputs (94) → hidden layer (e.g. 64 neurons, ReLU) → … → output neuron (1)
```

- Each `Linear` layer computes weighted sums (`w·x + b`); each neuron has its own weights.
- The **activation** (ReLU / tanh / sigmoid) adds the non-linearity. Without it, many layers would be
  no better than one (remember XOR).
- **Dropout** (Step 7) randomly switches off some neurons during training, so the network cannot rely on a
  few of them. It is a way to fight overfitting.
- The last neuron gives one **score**. `sigmoid(score)` = probability that the row is an attack.

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


def build_mlp(n_inputs, hidden_sizes, activation, dropout):
    layers, width = [], n_inputs
    for size in hidden_sizes:
        layers.append(nn.Linear(width, size))           # weights + bias
        layers.append(ACTIVATIONS[activation]())         # non-linearity
        if dropout > 0:
            layers.append(nn.Dropout(dropout))           # only active while training
        width = size
    layers.append(nn.Linear(width, 1))                   # output: one score per row
    return nn.Sequential(*layers)


example = build_mlp(X["train"].shape[1], [64], "relu", 0.0)
print(example)
print("weights + biases to learn:", sum(p.numel() for p in example.parameters()))

## 5. Training: the loop

One **epoch** = one pass over all training rows, in small **batches**. For every batch:

1. **forward pass**: the network scores the rows;
2. **loss**: how wrong the scores are. We use *binary cross-entropy*: small when the predicted probability of
   the true class is high, large when it is low;
3. **backpropagation**: PyTorch computes the gradient of the loss for every weight (the chain rule from Week 1);
4. **update**: the **optimizer** moves every weight a small step against its gradient. The **learning rate**
   sets the step size.
   - `sgd`: plain gradient descent, `w ← w − learning_rate × gradient`;
   - `sgd + momentum`: also keeps 90% of the previous step, so it speeds up in a steady direction;
   - `adam`: adapts the step size for each weight from its recent gradients; usually works with lr 0.001.

After each epoch we compute the **validation loss**. The lecturer's rule from Week 2: when the validation
curve stops going down and starts going up, the network is **overfitting**, so save the weights from the best
point and stop. That is **early stopping**: stop after `patience` epochs without a better validation loss, and
go back to the best weights. `patience = 0` turns it off (train all epochs, keep the last weights).

**Weight decay** (L2 regularisation, a *penalty* on large weights) pulls every weight a little towards 0 at
each step. Another way to fight overfitting.

In [ ]:
def train_model(model, X, y, optimizer_name, learning_rate, batch_size, epochs, patience,
                weight_decay=0.0, train_fraction=1.0):
    torch.manual_seed(SEED)
    X_train, y_train = X["train"], y["train"]
    if train_fraction < 1:                                      # use only part of the training rows
        keep = torch.randperm(len(y_train))[: int(len(y_train) * train_fraction)]
        X_train, y_train = X_train[keep], y_train[keep]

    loss_fn = nn.BCEWithLogitsLoss()                            # binary cross-entropy (sigmoid included)
    if optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "sgd + momentum":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)

    history = {"train_loss": [], "val_loss": []}
    best_val_loss, best_weights, best_epoch, waited = float("inf"), None, 0, 0
    print_every = max(1, epochs // 20)
    for epoch in range(1, epochs + 1):
        # --- train on all batches ---
        model.train()
        order = torch.randperm(len(y_train))                    # new random order every epoch
        total_loss = 0.0
        for start in range(0, len(y_train), batch_size):
            batch = order[start : start + batch_size]
            scores = model(X_train[batch]).squeeze(1)           # 1. forward pass
            loss = loss_fn(scores, y_train[batch])              # 2. loss
            optimizer.zero_grad()
            loss.backward()                                     # 3. backpropagation: gradients
            optimizer.step()                                    # 4. update the weights
            total_loss += loss.item() * len(batch)
        train_loss = total_loss / len(y_train)

        # --- check on the validation set (no learning here) ---
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X["val"]).squeeze(1), y["val"]).item()
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        # --- early stopping: remember the best weights ---
        improved = val_loss < best_val_loss
        if improved:
            best_val_loss, best_weights, best_epoch, waited = val_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            waited += 1
        if epoch % print_every == 0 or epoch == 1:
            print(f"epoch {epoch:3d}   train loss {train_loss:.4f}   val loss {val_loss:.4f}" + ("   ← best" if improved else ""))
        if patience and waited >= patience:
            print(f"early stopping at epoch {epoch}: no better val loss for {patience} epochs")
            break

    if patience:
        model.load_state_dict(best_weights)                     # go back to the best epoch
        print(f"kept the weights from epoch {best_epoch} (val loss {best_val_loss:.4f})")
    else:
        print(f"kept the last weights (epoch {epoch}); the best val loss was at epoch {best_epoch}")
    return history, best_epoch

## 6. Testing: how good is it?

The finished model scores the **test** rows (never seen during training or stopping). A probability
≥ 0.5 means "attack". The **confusion matrix** counts the four possible outcomes:

| | predicted normal | predicted attack |
| --- | --- | --- |
| **really normal** | correct | **false alarm** |
| **really attack** | **missed attack** | correct (detected) |

In [ ]:
def test_model(model, X_test, y_test, threshold=0.5):
    model.eval()
    with torch.no_grad():
        probability = torch.sigmoid(model(X_test).squeeze(1)).numpy()
    predicted = probability >= threshold
    attack = y_test.numpy() == 1
    tn, fp = int((~predicted & ~attack).sum()), int((predicted & ~attack).sum())
    fn, tp = int((~predicted & attack).sum()), int((predicted & attack).sum())
    return {"detection rate %": 100 * tp / (tp + fn), "false alarm rate %": 100 * fp / (fp + tn),
            "accuracy %": 100 * (tp + tn) / len(attack), "confusion": [[tn, fp], [fn, tp]]}


def show_run(name, history, best_epoch, result):
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.4, 1]})
    epochs = range(1, len(history["train_loss"]) + 1)
    left.plot(epochs, history["train_loss"], "--", color="gray", label="train loss")
    left.plot(epochs, history["val_loss"], color="green", label="validation loss")
    left.axvline(best_epoch, color="black", linestyle=":", label=f"best val loss (epoch {best_epoch})")
    left.set_yscale("log")
    left.set(title="Loss per epoch (lower is better)", xlabel="epoch", ylabel="loss (log scale)")
    left.legend()

    matrix = np.array(result["confusion"])
    right.imshow(matrix / matrix.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            share = matrix[i, j] / matrix[i].sum()
            right.text(j, i, f"{matrix[i, j]:,}\n{share:.2%}", ha="center", va="center",
                       color="white" if share > 0.5 else "black", fontsize=12)
    right.set_xticks([0, 1], ["normal", "attack"])
    right.set_yticks([0, 1], ["normal", "attack"])
    right.set(title="Test set: confusion matrix", xlabel="predicted", ylabel="true")
    fig.suptitle(f"{name}:  detected {result['detection rate %']:.2f}% of attacks,  "
                 f"false alarms {result['false alarm rate %']:.2f}% of normal traffic", fontweight="bold")
    fig.tight_layout()
    plt.show()


results = {}   # every experiment you run is kept here (Step 8 compares them)

## 7. Experiment ⭐

Change the settings below, then run this cell again (▶ or *Ctrl/Cmd + Enter*).
Give each experiment its own `run_name`, so Step 8 can compare them. Change **one setting at a time**,
otherwise you cannot tell which change caused the difference.

The first settings are the **baseline**. Suggested experiments are in **Step 9**.

In [ ]:
# @title Settings  { display-mode: "both" }
run_name = "baseline"        # @param {type:"string"}
scaling = "standard"         # @param ["standard", "log + standard", "none"]
hidden_layers = "64"         # @param {type:"string"}
activation = "relu"          # @param ["relu", "tanh", "sigmoid"]
dropout = 0.0                # @param {type:"slider", min:0.0, max:0.5, step:0.1}
optimizer_name = "adam"      # @param ["adam", "sgd", "sgd + momentum"]
learning_rate = 0.001        # @param [0.00001, 0.0001, 0.001, 0.01, 0.1] {type:"raw"}
batch_size = 256             # @param [32, 256, 4096] {type:"raw"}
epochs = 30                  # @param {type:"slider", min:1, max:150, step:1}
patience = 5                 # @param {type:"slider", min:0, max:20, step:1}
weight_decay = 0.0           # @param [0.0, 0.0001, 0.001, 0.01] {type:"raw"}
train_fraction = 1.0         # @param [1.0, 0.1, 0.01] {type:"raw"}

# hidden_layers: "64" = one hidden layer of 64 neurons; "128, 64" = two layers; "" = no hidden layer
hidden_sizes = [int(n) for n in hidden_layers.replace(",", " ").split()]
X, y = prepare_data(scaling)
torch.manual_seed(SEED)                                   # same starting weights every time
model = build_mlp(X["train"].shape[1], hidden_sizes, activation, dropout)
history, best_epoch = train_model(model, X, y, optimizer_name, learning_rate, batch_size, epochs,
                                  patience, weight_decay, train_fraction)
result = test_model(model, X["test"], y["test"])
results[run_name] = {"settings": dict(scaling=scaling, hidden=hidden_layers, activation=activation,
                                      dropout=dropout, optimizer=optimizer_name, lr=learning_rate,
                                      batch=batch_size, epochs=epochs, patience=patience,
                                      weight_decay=weight_decay, train_fraction=train_fraction),
                     "history": history, "best_epoch": best_epoch, **result}
show_run(run_name, history, best_epoch, result)

**How to read the loss chart** (Week 2):
- both curves go down and stay close → **good fit**;
- both stay high → **underfitting** (too small a network, too few epochs, learning rate too small …);
- train keeps going down but validation turns **up** → **overfitting**: the network learns the training
  rows by heart instead of the general pattern. The dotted line marks the best epoch.

## 8. Compare your experiments

A table of every run so far, and their validation curves on one chart.
To compare only some runs, write their names, e.g. `compare(["baseline", "lr 0.1"])`.

In [ ]:
def compare(names=None):
    names = names or list(results)
    table = pd.DataFrame({
        name: {**results[name]["settings"],
               "epochs run": len(results[name]["history"]["val_loss"]),
               "best epoch": results[name]["best_epoch"],
               "best val loss": round(min(results[name]["history"]["val_loss"]), 4),
               "detected %": round(results[name]["detection rate %"], 2),
               "false alarms %": round(results[name]["false alarm rate %"], 2)}
        for name in names}).T
    display(table)

    plt.figure(figsize=(10, 4.5))
    for name in names:
        plt.plot(range(1, len(results[name]["history"]["val_loss"]) + 1), results[name]["history"]["val_loss"], label=name)
    plt.yscale("log")
    plt.title("Validation loss per epoch (lower is better)")
    plt.xlabel("epoch")
    plt.ylabel("val loss (log scale)")
    plt.legend()
    plt.show()


compare()

## 9. Suggested experiments

Each experiment changes **one** setting of the baseline in Step 7 (then run Step 7 again, with a new
`run_name`). **Guess first** (先猜再跑), then run, then compare with the numbers below.
The groups fit the report and slides well, and each group member can take one.

"We saw" = one run on a laptop CPU, seed 42. Colab may differ slightly in the last digits; the trend is
what matters. **Baseline**: best val loss 0.0076, attacks detected 99.84%, false alarms 1.28%.

### A. Scaling (normalisation)

| change | best val loss | detected | false alarms | what happened |
| --- | --- | --- | --- | --- |
| `scaling = "none"` | 7.3 | 96.6% | 1.2% | first-epoch train loss ≈ 45,000: huge inputs → huge weighted sums |
| baseline (`standard`) | 0.0076 | 99.8% | 1.3% | |
| `scaling = "log + standard"` | **0.0058** | 99.8% | 0.9% | best of the three |

### B. Learning rate (step size)

| `learning_rate` | best val loss | false alarms | what happened |
| --- | --- | --- | --- |
| 0.00001 | 0.113 | 35% | too small: still learning after 30 epochs |
| 0.0001 | 0.028 | 5.5% | slow |
| 0.001 (baseline) | 0.0076 | 1.3% | |
| 0.01 | 0.0097 | 1.8% | fast at first, stops at epoch 24 |
| 0.1 | 0.061 | 16% | too big: val loss jumps around, stops at epoch 7 |

### C. Optimizer (all at learning rate 0.001, then SGD at 0.1)

| change | best val loss | false alarms |
| --- | --- | --- |
| `optimizer_name = "sgd"` | 0.100 | 35% |
| `"sgd + momentum"` | 0.047 | 6.7% |
| `"adam"` (baseline) | 0.0076 | 1.3% |
| `"sgd"` with `learning_rate = 0.1` | 0.017 | 2.1% |

Plain SGD needs a much bigger learning rate. Adam adapts the step for every weight, so 0.001 works.
The best learning rate depends on the optimizer.

### D. Network size (capacity)

| `hidden_layers` | weights | best val loss | false alarms |
| --- | --- | --- | --- |
| `""` (no hidden layer = one neuron, like the Week 1 perceptron with a sigmoid) | 95 | 0.055 | 7.5% |
| `"8"` | 769 | 0.017 | 2.4% |
| `"64"` (baseline) | 6,145 | 0.0076 | 1.3% |
| `"128, 64"` | 20,481 | 0.0072 | 0.8% |

### E. Overfitting and how to stop it ⭐ (Week 2)

With all 82,545 training rows this network hardly overfits. To *see* overfitting, give a big network
little data and many epochs:

> `train_fraction = 0.01` (825 rows), `hidden_layers = "256, 128"`, `epochs = 150`, `patience = 0`

We saw: train loss falls to 0.001, but **validation loss is lowest at epoch 7 (0.090) and then rises to
0.34**. The network learns the 825 rows by heart. Now keep these settings and change one thing:

| fix | val loss at the end | note |
| --- | --- | --- |
| nothing | 0.34 | overfitting |
| `patience = 5` (early stopping) | stops at epoch 12, keeps epoch 7: 0.090 | the lecturer's rule: save the best weights, stop |
| `dropout = 0.3` / `0.5` | 0.25 / 0.19 | helps a little |
| `weight_decay = 0.001` / `0.01` | 0.084 / **0.075** | strongest fix here |
| `hidden_layers = "16"` (smaller network) | 0.108 | no overfitting, but it learns less |
| `train_fraction = 0.1` (10× more data) | 0.044 → 0.099 | more data: much lower loss, still overfits later |

### F. Extra: vanishing gradients with sigmoid

> `hidden_layers = "64, 64, 64, 64"`, `optimizer_name = "sgd"`, `learning_rate = 0.01`

With `activation = "sigmoid"` the loss stays at 0.33 for all 30 epochs and the model calls **every** row an
attack (100% false alarms). The sigmoid's slope is at most 0.25 and backpropagation multiplies it once per
layer, so the early layers hardly change. With `"relu"`: val loss 0.026, false alarms 3.3%.

### For the report

For each group: the Step 8 table + the validation loss chart, one sentence on what changed, and one
sentence on *why* (the course idea behind it). Only one seed was run, so small differences (e.g. 1.2% vs
1.3% false alarms) may be random; say so.